In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nusratjahannawal/banglapsg-v2/BanglaPSG-Grievance-Dataset_V2.xlsx


In [2]:
import re, unicodedata, difflib
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

SRC = "/kaggle/input/datasets/nusratjahannawal/banglapsg-v2/BanglaPSG-Grievance-Dataset_V2.xlsx"
OUT = "/kaggle/working"
TFIDF_MIN = 0.80      # shortlist threshold
RATIO_MIN = 0.90      # final fuzzy threshold
VALID_SEV = {"Low", "Medium", "High"}

raw = pd.read_excel(SRC)
df = raw.rename(columns={"No.": "Source_No", "Grievance Text": "Text"}).copy()
df.insert(0, "Original_No", range(1, len(df) + 1))   # file row order, 1..6105

assert df["Original_No"].is_unique
assert df["Text"].notna().all() and df["Category"].notna().all() and df["Severity"].notna().all()
print(df.shape, df.columns.tolist())
print("Source_No duplicated values:", df["Source_No"].duplicated(keep=False).sum(), "rows")

(6105, 5) ['Original_No', 'Source_No', 'Text', 'Category', 'Severity']
Source_No duplicated values: 6 rows


In [3]:
df["Text"] = df["Text"].astype(str).map(lambda t: unicodedata.normalize("NFC", t))
for c in ["Category", "Severity"]:
    df[c] = df[c].astype(str).str.strip()

print(df["Category"].value_counts(), "\n")
print(df["Severity"].value_counts(), "\n")
assert set(df["Severity"]) <= VALID_SEV, "Unexpected Severity label found"
print("Empty text:", (df["Text"].str.strip() == "").sum(),
      "| Very short (<10 chars):", (df["Text"].str.len() < 10).sum())
print("Text starting with '=' (Excel formula risk):", df["Text"].str.startswith("=").sum())

Category
Road & Transport    1061
Food                1036
Waste               1015
Healthcare          1015
Electricity         1007
Water                971
Name: count, dtype: int64 

Severity
Medium    2464
Low       1988
High      1653
Name: count, dtype: int64 

Empty text: 0 | Very short (<10 chars): 1
Text starting with '=' (Excel formula risk): 0


In [4]:
def norm_key(t):
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", t)   #zero-width chars and BOM
    t = t.replace("\\n", " ")
    t = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", t)         #punctuation and newlines -> space
    return re.sub(r"\s+", " ", t).strip().lower()

df["Key_Spaced"] = df["Text"].map(norm_key)
df["Key"] = df["Key_Spaced"].str.replace(" ", "", regex=False)
assert (df["Key"] == "").sum() == 0, "Some texts became empty after normalization"
print("Keys OK")

Keys OK


In [5]:
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
X = vec.fit_transform(df["Key_Spaced"])
n = X.shape[0]

cands, CH = [], 500
for s in range(0, n, CH):
    sim = (X[s:s + CH] @ X.T).toarray()
    rows, cols = np.where(sim >= TFIDF_MIN)
    for r, c in zip(rows, cols):
        if c > s + r:
            cands.append((s + r, c, sim[r, c]))
print("TF-IDF candidates:", len(cands))

fuzzy = []
for i, j, cos in cands:
    a, b = df["Key"].iat[i], df["Key"].iat[j]
    r = 1.0 if a == b else difflib.SequenceMatcher(None, a, b, autojunk=False).ratio()
    if r >= RATIO_MIN:
        fuzzy.append((i, j, cos, r))
print("Confirmed pairs (ratio >= %.2f): %d" % (RATIO_MIN, len(fuzzy)))

TF-IDF candidates: 171
Confirmed pairs (ratio >= 0.90): 140


In [6]:
parent = list(range(n))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[ry] = rx

for col in ["Text", "Key"]:
    for _, idx in df.groupby(col).groups.items():
        idx = list(idx)
        for k in idx[1:]:
            union(idx[0], k)
for i, j, _, _ in fuzzy:
    union(i, j)

df["root"] = [find(i) for i in range(n)]
dup = df[df.groupby("root")["root"].transform("size") > 1].copy()

def ctype(g):
    if g["Text"].nunique() == 1: return "exact"
    if g["Key"].nunique() == 1:  return "normalized"
    return "fuzzy"

cl = pd.DataFrame([
    dict(root=r, n=len(g), type=ctype(g), n_sev=g["Severity"].nunique(), n_cat=g["Category"].nunique())
    for r, g in dup.groupby("root")
]).sort_values("root").reset_index(drop=True)
cl["Cluster_ID"] = ["C%04d" % (k + 1) for k in range(len(cl))]
cl["Resolution"] = np.where((cl["type"] != "fuzzy") & (cl["n_sev"] == 1) & (cl["n_cat"] == 1),
                            "auto", "manual")

print(cl.groupby(["type", "Resolution"]).size().unstack(fill_value=0))
print("Total clusters:", len(cl), "| rows in clusters:", len(dup))
print("Manual clusters:", (cl["Resolution"] == "manual").sum())
ex = cl[cl["type"] == "exact"]
print("Exact-text clusters:", len(ex), "| with severity conflict:", (ex["n_sev"] > 1).sum(),
      "| with category conflict:", (ex["n_cat"] > 1).sum())

Resolution  auto  manual
type                    
exact         56      40
fuzzy          0      14
normalized     9      10
Total clusters: 129 | rows in clusters: 264
Manual clusters: 64
Exact-text clusters: 96 | with severity conflict: 40 | with category conflict: 1


In [7]:
mem = dup.merge(cl[["root", "Cluster_ID", "type", "Resolution"]], on="root")
mem = mem.sort_values(["Cluster_ID", "Original_No"])

rev = mem[mem["Resolution"] == "manual"][
    ["Cluster_ID", "type", "Original_No", "Category", "Severity", "Text"]
].rename(columns={"type": "Cluster_Type"})
for c in ["Is_Duplicate", "Keep_Original_No", "Final_Category", "Final_Severity", "Reason"]:
    rev[c] = ""

assert rev["Cluster_ID"].nunique() == (cl["Resolution"] == "manual").sum()

fz_df = pd.DataFrame(
    [(df["Original_No"].iat[i], df["Original_No"].iat[j], c, r) for i, j, c, r in fuzzy],
    columns=["No_A", "No_B", "Cosine", "Ratio"])

with pd.ExcelWriter(f"{OUT}/BanglaPSG_Review_Sheet.xlsx") as w:
    rev.to_excel(w, sheet_name="review", index=False)
    df[["Original_No", "Source_No", "Text", "Category", "Severity"]].to_excel(w, sheet_name="base_clean", index=False)
    mem[["Original_No", "Cluster_ID", "type", "Resolution"]].to_excel(w, sheet_name="cluster_map", index=False)
    fz_df.to_excel(w, sheet_name="fuzzy_pairs", index=False)

print("Review rows:", len(rev), "| clusters to decide:", rev["Cluster_ID"].nunique())

Review rows: 133 | clusters to decide: 64
